In [ ]:
# Huggingface Transformers Quicktour
# https://github.com/huggingface/notebooks/blob/main/transformers_doc/en/quicktour.ipynb
# https://huggingface.co/learn/nlp-course/chapter1/4?fw=pt
# https://huggingface.co/docs/transformers/en/task_summary
# https://huggingface.co/docs/transformers/notebooks
#+# https://www.youtube.com/watch?v=bCz4OMemCcA&ab_channel=UmarJamil

In [ ]:
!pip install transformers
!pip install transformers datasets

In [ ]:
!pip install torch
!pip install tensorflow

In [3]:
from transformers import pipeline

In [4]:
# test sentiment analysis
classifier = pipeline("sentiment-analysis")
samples1 = ["We are very happy to show you the Transformers library.", "We hope you don't hate it."]
samples2= ["I find transformers interesting but hard to understand.", "I would kind of venture to watch the solar eclipse."]
results = classifier(samples2)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [5]:
print(results)
for item in results:
    print(item)
    print(item['label'])
    print(item['score'])

[{'label': 'NEGATIVE', 'score': 0.9949429631233215}, {'label': 'POSITIVE', 'score': 0.9747183322906494}]
{'label': 'NEGATIVE', 'score': 0.9949429631233215}
NEGATIVE
0.9949429631233215
{'label': 'POSITIVE', 'score': 0.9747183322906494}
POSITIVE
0.9747183322906494


In [6]:
for result in results:
    print(f"label: {result['label']}, with score: {round(result['score'], 4)}")

label: NEGATIVE, with score: 0.9949
label: POSITIVE, with score: 0.9747


In [7]:
# try out speech recognition
import torch
from transformers import pipeline

speech_recognizer = pipeline("automatic-speech-recognition", model="facebook/wav2vec2-base-960h")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-960h
Key                        | Status  | 
---------------------------+---------+-
wav2vec2.masked_spec_embed | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

In [8]:
from datasets import load_dataset, Audio

dataset = load_dataset("PolyAI/minds14", name="en-US", split="train")
dataset = dataset.cast_column("audio", Audio(sampling_rate=speech_recognizer.feature_extractor.sampling_rate))

README.md: 0.00B [00:00, ?B/s]

en-US/train-00000-of-00001.parquet:   0%|          | 0.00/34.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/563 [00:00<?, ? examples/s]

In [9]:
result = speech_recognizer(dataset[:4]["audio"])
print([d["text"] for d in result])

['I WOULD LIKE TO SET UP A JOINT ACCOUNT WITH MY PARTNER HOW DO I PROCEED WITH DOING THAT', "FONDERING HOW I'D SET UP A JOIN TO HELL T WITH MY WIFE AND WHERE THE AP MIGHT BE", "I I'D LIKE TOY SET UP A JOINT ACCOUNT WITH MY PARTNER I'M NOT SEEING THE OPTION TO DO IT ON THE APSO I CALLED IN TO GET SOME HELP CAN I JUST DO IT OVER THE PHONE WITH YOU AND GIVE YOU THE INFORMATION OR SHOULD I DO IT IN THE AP AN I'M MISSING SOMETHING UQUETTE HAD PREFERRED TO JUST DO IT OVER THE PHONE OF POSSIBLE THINGS", 'HOW DO I FURN A JOINA COUT']


In [10]:
# Load pretrained models and then save them

from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer
model_name = "nlptown/bert-base-multilingual-uncased-sentiment"
pt_model = AutoModelForSequenceClassification.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [12]:
samples = ["We are very happy to show you the Transformers library.", "We hope you don't hate it."]

pt_batch = tokenizer(
    samples,
    padding=True,
    truncation=True,
    max_length=512,
    return_tensors="pt",
)

tf_batch = tokenizer(
    samples,
    padding=True,
    truncation=True,
    max_length=512,
    return_tensors = "pt",
)

In [13]:
from torch import nn

pt_outputs = pt_model(**pt_batch)
pt_predictions = nn.functional.softmax(pt_outputs.logits, dim=-1)
print(pt_predictions)

tensor([[0.0022, 0.0019, 0.0131, 0.2332, 0.7496],
        [0.2084, 0.1826, 0.1969, 0.1755, 0.2365]], grad_fn=<SoftmaxBackward0>)


In [14]:
from google.colab import drive
drive.mount('/content/drive')
#change this based on your setup
root = '/content/drive/MyDrive/Colab/ML/'
modelpath =  root + 'models/'

Mounted at /content/drive


In [15]:
# Save the model
pt_save_directory = modelpath
tokenizer.save_pretrained(pt_save_directory)
pt_model.save_pretrained(pt_save_directory)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]